In [37]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

SEED = 42                                   
DATA_PATH = Path('../data/METABRIC_RNA_Mutation.csv')

In [38]:
header = pd.read_csv(DATA_PATH, nrows=0).columns
mut_cols = [c for c in header if c.endswith('_mut')]
print("Number of _mut columns:", len(mut_cols))

df = pd.read_csv(DATA_PATH, dtype={c: str for c in mut_cols})
print("Shape:", df.shape)

Number of _mut columns: 173
Shape: (1904, 693)


In [39]:
mut_values = pd.Series(df[mut_cols].to_numpy().ravel())
vc = mut_values.value_counts(dropna=False)
print("\nDistinct raw values in mut block:", len(vc))
print(vc.head(10))

zero_like = [v for v in vc.index if isinstance(v, str) and v.strip() in {'0', '0.0', '0.00'}]
print("\nZero-like spellings found:", zero_like)


Distinct raw values in mut block: 7668
0              319653
H1047R            285
E545K             127
E542K              73
E17K               63
X308_splice        59
N345K              42
K700E              36
R175H              34
V2061L             34
Name: count, dtype: int64

Zero-like spellings found: ['0']


In [40]:
print("Total NaN in mut block:", int(df[mut_cols].isna().sum().sum()))
is_mut = df[mut_cols].notna() & ~df[mut_cols].isin(zero_like)
mut_counts = is_mut.sum().sort_values(ascending=False)
print("\nTop 10 genes by number of mutated patients:")
print(mut_counts.head(10))
print("\nColumns with 0 mutated patients:", int((mut_counts == 0).sum()))
print("Columns with < 10 mutated patients:", int((mut_counts < 10).sum()))

for c in ['rasgef1b_mut', 'hras_mut', 'smarcb1_mut', 'siah1_mut']:
    print(c, df[c].value_counts(dropna=False).to_dict())


Total NaN in mut block: 0

Top 10 genes by number of mutated patients:
pik3ca_mut    795
tp53_mut      659
muc16_mut     326
ahnak2_mut    311
kmt2c_mut     234
syne1_mut     232
gata3_mut     230
map3k1_mut    198
ahnak_mut     176
dnah11_mut    175
dtype: int64

Columns with 0 mutated patients: 0
Columns with < 10 mutated patients: 23
rasgef1b_mut {'0': 1897, 'V418A': 1, 'S343G': 1, 'R140Q': 1, 'X2_splice': 1, 'R353H': 1, 'H396Y': 1, 'E78K': 1}
hras_mut {'0': 1902, 'Q61R': 1, 'N86T': 1}
smarcb1_mut {'0': 1902, 'D159N': 1, 'I28L': 1}
siah1_mut {'0': 1903, 'T249A': 1}


In [41]:
raw = df[mut_cols]

is_zero = (raw == '0')
n_total   = raw.size
n_zero    = int(is_zero.to_numpy().sum())
n_nonzero = n_total - n_zero
print("total cells:", n_total, "| zero:", n_zero, "| non-zero:", n_nonzero)

mut_bin = (~is_zero).astype('uint8')
assert int(mut_bin.to_numpy().sum()) == n_nonzero, "binary count mismatch"
assert n_zero == 319653, "zero count differs from Step 1"
print("OK: mut_bin matches. shape:", mut_bin.shape)

vals = pd.Series(raw.to_numpy()[~is_zero.to_numpy()])
print("non-zero values:", len(vals))

has_space = vals.str.contains(' ', regex=False)
print("cells with a space:", int(has_space.sum()))
print(vals[has_space].value_counts().head(15))

print("\nSample of non-space values:")
print(vals[~has_space].sample(10, random_state=SEED).tolist())

total cells: 329392 | zero: 319653 | non-zero: 9739
OK: mut_bin matches. shape: (1904, 173)
non-zero values: 9739
cells with a space: 728
H1047R R108H                    5
R1344Q I2386V                   5
E542K E726K                     5
H1047R H1048R                   4
E545K E726K                     4
H1047R E726K                    4
H1047R E453K                    3
S3366Mfs*2 V3365_S3366insELY    3
Q573P R1378Q                    3
E545K M1004I                    2
H1047R G118D                    2
S3603R A1115T                   2
H1047R E365K                    2
H1047R P104L                    2
H1047R E81K                     2
Name: count, dtype: int64

Sample of non-space values:
['P744Qfs*26', 'S1078Y', 'E404K', 'E545K', 'Y2405N', 'G2282R', 'L134R', 'V4927L', 'E2550K', 'L139F']


In [42]:
burden = mut_bin.sum(axis=1)
print("\nPer-patient burden (n mutated genes out of 173):")
print(burden.describe())
print("Patients with burden == 0:", int((burden == 0).sum()))

print("\nSpearman(burden, mutation_count):",
      round(burden.corr(df['mutation_count'], method='spearman'), 3))

n_mut_pat = mut_bin.sum()
for k in [5, 10, 19, 38]:       # 19 ~ 1%, 38 ~ 2% of 1904
    print(f"\ngenes mutated in < {k} patients: {(n_mut_pat < k).sum()}")


Per-patient burden (n mutated genes out of 173):
count    1904.000000
mean        5.115021
std         3.351521
min         0.000000
25%         3.000000
50%         5.000000
75%         7.000000
max        48.000000
dtype: float64
Patients with burden == 0: 45

Spearman(burden, mutation_count): 0.978

genes mutated in < 5 patients: 11

genes mutated in < 10 patients: 23

genes mutated in < 19 patients: 54

genes mutated in < 38 patients: 102


In [43]:
time = df['overall_survival_months']
event = (df['overall_survival'] == 0).astype(int)
print("Before drop -> events", int(event.sum()), "| censored:", int((event == 0).sum()))

zero_mask = time <= 0
print("\nPatients with time <= 0:", int(zero_mask.sum()))
print(df.loc[zero_mask, ['patient_id', 'overall_survival',
                         'overall_survival_months', 'death_from_cancer']])

Before drop -> events 1103 | censored: 801

Patients with time <= 0: 1
     patient_id  overall_survival  overall_survival_months death_from_cancer
170         284                 1                      0.0            Living


In [44]:
keep = ~zero_mask
df_clean = df.loc[keep].reset_index(drop=True)
mut_bin  = mut_bin.loc[keep].reset_index(drop=True)
y = pd.DataFrame({'time': time[keep].to_numpy(),
                  'event': event[keep].to_numpy()})

assert len(df_clean) == len(mut_bin) == len(y)
assert (y['time'] > 0).all()
print("\nAfter drop -> n:", len(y),
      "| events:", int(y['event'].sum()),
      "| censored:", int((y['event'] == 0).sum()))


After drop -> n: 1903 | events: 1103 | censored: 800


In [45]:
all_cols = list(df_clean.columns)

mut_set = set(mut_cols)

last_clinical_idx = all_cols.index('death_from_cancer')
clinical_block = all_cols[:last_clinical_idx + 1]

expr_block = [c for c in all_cols[last_clinical_idx + 1:] if c not in mut_set]

print("clinical block :", len(clinical_block))
print("expression blk :", len(expr_block))
print("mutation block :", len(mut_cols))
print("total          :", len(clinical_block) + len(expr_block) + len(mut_cols), "(expect 693)")


non_numerical_expr = [c for c in expr_block if not pd.api.types.is_numeric_dtype(df_clean[c])]
print('\n')
print("non-numeric in expression block (expect []):", non_numerical_expr)
print("NaN in expression block (expect 0):", int(df_clean[expr_block].isna().sum().sum()))

clinical block : 31
expression blk : 489
mutation block : 173
total          : 693 (expect 693)


non-numeric in expression block (expect []): []
NaN in expression block (expect 0): 0


In [46]:
rows = []

for c in clinical_block:
    s = df_clean[c]
    rows.append({
        'column' : c,
        'dtype' : str(s.dtype),
        'missing_%' : round(s.isna().mean() * 100, 1),
        'n_unique' : s.nunique(dropna=True),
        'examples': s.dropna().unique()[:3].tolist(),
    })

clin_audit = pd.DataFrame(rows)
pd.set_option('display.max_rows', 100, 'display.max_colwidth', 60, 'display.width', 200)
print(clin_audit.to_string(index=False))

                        column   dtype  missing_%  n_unique                                                                                                         examples
                    patient_id   int64        0.0      1903                                                                                                        [0, 2, 5]
              age_at_diagnosis float64        0.0      1572                                                                                            [75.65, 43.19, 48.87]
        type_of_breast_surgery  object        1.1         2                                                                                  [MASTECTOMY, BREAST CONSERVING]
                   cancer_type  object        0.0         1                                                                                                  [Breast Cancer]
          cancer_type_detailed  object        0.8         6 [Breast Invasive Ductal Carcinoma, Breast Mixed Ductal and Lobular Carcinom

In [47]:
LEAKAGE_COLS = ['patient_id', 'overall_survival',
                'overall_survival_months', 'death_from_cancer']

TREATMENT_COLS = ['chemotherapy', 'hormone_therapy', 'radio_therapy']
REFERENCE_COLS = ['nottingham_prognostic_index']          
META_COLS = ['cohort']                                    
DROP_REDUNDANT = ['cancer_type', 'er_status_measured_by_ihc',
                  'oncotree_code', 'tumor_other_histologic_subtype']

CLINICAL_FEATS = ['age_at_diagnosis', 'type_of_breast_surgery', 'cancer_type_detailed',
                  'cellularity', 'er_status', 'her2_status', 'pr_status',
                  'neoplasm_histologic_grade', 'inferred_menopausal_state',
                  'primary_tumor_laterality', 'lymph_nodes_examined_positive',
                  'tumor_size', 'tumor_stage']

MOLECULAR_CLIN_BLOCK = ['mutation_count', 'pam50_+_claudin-low_subtype', 'integrative_cluster',
                        '3-gene_classifier_subtype', 'her2_status_measured_by_snp6']

MOLECULAR_FEATS = expr_block + mut_cols + MOLECULAR_CLIN_BLOCK
COMBINED_FEATS  = CLINICAL_FEATS + MOLECULAR_FEATS


groups = {
    'leakage': LEAKAGE_COLS, 'treatment': TREATMENT_COLS, 'reference': REFERENCE_COLS,
    'meta': META_COLS, 'redundant': DROP_REDUNDANT,
    'clinical': CLINICAL_FEATS, 'molecular_in_clin_block': MOLECULAR_CLIN_BLOCK,
}

assigned = [c for g in groups.values() for c in g]
print("Unassigned clinical-block cols (expect []):", [c for c in clinical_block if c not in assigned])
print("Assigned but not in clinical block (expect []):", [c for c in assigned if c not in clinical_block])
print("Duplicates (expect []):", sorted({c for c in assigned if assigned.count(c) > 1}))
print({k: len(v) for k, v in groups.items()})
print("Feature counts -> clinical:", len(CLINICAL_FEATS),
      "| molecular:", len(MOLECULAR_FEATS), "| combined:", len(COMBINED_FEATS))
assert not set(COMBINED_FEATS) & set(LEAKAGE_COLS + TREATMENT_COLS + REFERENCE_COLS + META_COLS)



Unassigned clinical-block cols (expect []): []
Assigned but not in clinical block (expect []): []
Duplicates (expect []): []
{'leakage': 4, 'treatment': 3, 'reference': 1, 'meta': 1, 'redundant': 4, 'clinical': 13, 'molecular_in_clin_block': 5}
Feature counts -> clinical: 13 | molecular: 667 | combined: 680


In [48]:
print("\n-- oncotree_code vs cancer_type_detailed --")
print(pd.crosstab(df_clean['cancer_type_detailed'], df_clean['oncotree_code'], dropna=False))
print("\n-- er_status vs er_status_measured_by_ihc --")
print(pd.crosstab(df_clean['er_status'], df_clean['er_status_measured_by_ihc'], dropna=False))
print("\n-- tumor_other_histologic_subtype vs cancer_type_detailed --")
print(pd.crosstab(df_clean['tumor_other_histologic_subtype'],
                  df_clean['cancer_type_detailed'], dropna=False))



d = df_clean[['tumor_size', 'lymph_nodes_examined_positive',
              'neoplasm_histologic_grade', 'nottingham_prognostic_index']].dropna()

node_score = np.select([d['lymph_nodes_examined_positive'] == 0,
                        d['lymph_nodes_examined_positive'] <= 3], [1, 2], default=3)


npi_hat = 0.002 * d['tumor_size'] + node_score + d['neoplasm_histologic_grade']
diff = (npi_hat - d['nottingham_prognostic_index']).abs()
print("NPI match within 0.05:", round(float((diff < 0.05).mean()), 3), "| rows:", len(d))
print("Max abs diff:", round(float(diff.max()), 3))


bad = d[diff >= 0.05].assign(npi_hat=npi_hat[diff >= 0.05])
print(bad.head(8))


print("\nNPI missing:", int(df_clean['nottingham_prognostic_index'].isna().sum()))


print("\n", df_clean['tumor_other_histologic_subtype'].value_counts(dropna=False))


-- oncotree_code vs cancer_type_detailed --
oncotree_code                              BREAST   IDC  ILC  IMMC  MBC  MDLC  NaN
cancer_type_detailed                                                              
Breast                                         17     0    0     0    0     0    0
Breast Invasive Ductal Carcinoma                0  1499    0     0    0     0    0
Breast Invasive Lobular Carcinoma               0     0  142     0    0     0    0
Breast Invasive Mixed Mucinous Carcinoma        0     0    0    22    0     0    0
Breast Mixed Ductal and Lobular Carcinoma       0     0    0     0    0   207    0
Metaplastic Breast Cancer                       0     0    0     0    1     0    0
NaN                                             0     0    0     0    0     0   15

-- er_status vs er_status_measured_by_ihc --
er_status_measured_by_ihc  Negative  Positve  NaN
er_status                                        
Negative                        375       64    5
Positive    

In [49]:
indices = np.arange(len(df_clean))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=SEED,
    stratify=y['event']
)

df_train = df_clean.iloc[train_idx].copy()
df_test = df_clean.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Training patients:", len(df_train))
print("Testing patients:", len(df_test))

print("\nTraining events:")
print(y_train['event'].value_counts().sort_index())

print("\nTesting events:")
print(y_test['event'].value_counts().sort_index())

print("\nPatients in both train and test:")
print(df_train['patient_id'].isin(df_test['patient_id']).sum())

Training patients: 1522
Testing patients: 381

Training events:
event
0    640
1    882
Name: count, dtype: int64

Testing events:
event
0    160
1    221
Name: count, dtype: int64

Patients in both train and test:
0


In [50]:

# Reset indexes so rows line up by position, then combine features and mutations

df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

mut_train = mut_bin.iloc[train_idx].reset_index(drop=True)
mut_test = mut_bin.iloc[test_idx].reset_index(drop=True)

# Remove raw mutation columns and add binary mutation data
X_train = pd.concat(
    [df_train.drop(columns=mut_cols), mut_train], axis=1
)

X_test = pd.concat(
    [df_test.drop(columns=mut_cols), mut_test], axis=1
)

# Check that the data is correctly aligned
assert X_train.shape == (1522, 693), X_train.shape
assert X_test.shape == (381, 693), X_test.shape

assert len(X_train) == len(y_train)
assert len(X_test) == len(y_test)

assert X_train[mut_cols].isna().sum().sum() == 0
assert X_test[mut_cols].isna().sum().sum() == 0

assert X_train['patient_id'].is_unique
assert X_test['patient_id'].is_unique

assert set(X_train['patient_id']).isdisjoint(
    set(X_test['patient_id'])
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Training outcomes:", y_train.shape)
print("Testing outcomes:", y_test.shape)
print("All checks passed.")


Training shape: (1522, 693)
Testing shape: (381, 693)
Training outcomes: (1522, 2)
Testing outcomes: (381, 2)
All checks passed.


In [51]:

raw_count = int(
    df_clean[mut_cols].astype(str).ne('0').sum().sum()
)

binary_count = int(mut_bin.to_numpy().sum())

split_count = int(
    X_train[mut_cols].to_numpy().sum()
    + X_test[mut_cols].to_numpy().sum()
)

# Check mutations in the patient removed due to zero survival time
removed_count = int(
    (df.loc[zero_mask, mut_cols].astype(str) != '0').sum().sum()
)

print("Mutations in cleaned data:", raw_count)
print("Mutations after binary conversion:", binary_count)
print("Mutations in train and test:", split_count)
print("Mutations in removed patient(s):", removed_count)
print("Original total minus removed:", n_nonzero - removed_count)

# Stop if the mutation counts do not match
assert raw_count == binary_count == split_count == n_nonzero - removed_count

# 2. Check the training and testing outcomes
for name, outcomes in [('Training', y_train), ('Testing', y_test)]:
    print(
        name,
        "| Patients:", len(outcomes),
        "| Deaths:", int(outcomes['event'].sum()),
        "| Event rate:", round(outcomes['event'].mean(), 3)
    )

# 3. Create folders for the saved files
Path('../configs').mkdir(parents=True, exist_ok=True)
Path('../data/processed').mkdir(parents=True, exist_ok=True)

# 4. Save the patient IDs used in each split
split_info = {
    'seed': SEED,
    'test_size': 0.20,
    'stratify': 'event',
    'train_patient_ids': X_train['patient_id'].tolist(),
    'test_patient_ids': X_test['patient_id'].tolist()
}

with open('../configs/split.json', 'w') as file:
    json.dump(split_info, file, indent=2)

# 5. Keep IDs and extra information separate from model features
extra_cols = (
    ['patient_id', 'cohort']
    + TREATMENT_COLS
    + REFERENCE_COLS
)

# Save the selected model features
X_train[COMBINED_FEATS].to_pickle(
    '../data/processed/train_features.pkl'
)
X_test[COMBINED_FEATS].to_pickle(
    '../data/processed/test_features.pkl'
)

# Save extra information separately
X_train[extra_cols].to_pickle(
    '../data/processed/train_aux.pkl'
)
X_test[extra_cols].to_pickle(
    '../data/processed/test_aux.pkl'
)

# Save survival time and event labels
y_train.to_pickle('../data/processed/y_train.pkl')
y_test.to_pickle('../data/processed/y_test.pkl')

# 6. Reload the training features and check the saved file
saved_features = pd.read_pickle(
    '../data/processed/train_features.pkl'
)

assert saved_features.shape == (1522, 680)

excluded_cols = (
    LEAKAGE_COLS
    + TREATMENT_COLS
    + REFERENCE_COLS
    + META_COLS
)

assert not set(saved_features.columns).intersection(excluded_cols)

print("Training features saved:", saved_features.shape)
print("All checks passed. Files saved successfully.")

print("Missing values in saved train features:", int(saved_features.isna().sum().sum()))


Mutations in cleaned data: 9737
Mutations after binary conversion: 9737
Mutations in train and test: 9737
Mutations in removed patient(s): 2
Original total minus removed: 9737
Training | Patients: 1522 | Deaths: 882 | Event rate: 0.58
Testing | Patients: 381 | Deaths: 221 | Event rate: 0.58
Training features saved: (1522, 680)
All checks passed. Files saved successfully.
Missing values in saved train features: 824


In [52]:
print("Train cohorts:")
print(X_train['cohort'].value_counts().sort_index())

print("\nTest cohorts:")
print(X_test['cohort'].value_counts().sort_index())

Train cohorts:
cohort
1.0    382
2.0    230
3.0    584
4.0    190
5.0    136
Name: count, dtype: int64

Test cohorts:
cohort
1.0     98
2.0     56
3.0    150
4.0     46
5.0     31
Name: count, dtype: int64


In [53]:
cols = CLINICAL_FEATS + MOLECULAR_CLIN_BLOCK

for c in cols:
    print("\nColumn:", c)
    print("Type:", X_train[c].dtype)
    print("Missing:", X_train[c].isna().sum())
    print("Unique Values:", X_train[c].nunique())

    if X_train[c].nunique() <= 12:
        print(X_train[c].value_counts(dropna=False))
    else:
        print(X_train[c].describe())          


Column: age_at_diagnosis
Type: float64
Missing: 0
Unique Values: 1291
count    1522.000000
mean       60.781675
std        12.973667
min        26.360000
25%        51.157500
50%        61.315000
75%        70.497500
max        92.140000
Name: age_at_diagnosis, dtype: float64

Column: type_of_breast_surgery
Type: object
Missing: 16
Unique Values: 2
type_of_breast_surgery
MASTECTOMY           904
BREAST CONSERVING    602
NaN                   16
Name: count, dtype: int64

Column: cancer_type_detailed
Type: object
Missing: 12
Unique Values: 6
cancer_type_detailed
Breast Invasive Ductal Carcinoma             1207
Breast Mixed Ductal and Lobular Carcinoma     168
Breast Invasive Lobular Carcinoma             109
Breast Invasive Mixed Mucinous Carcinoma       15
NaN                                            12
Breast                                         10
Metaplastic Breast Cancer                       1
Name: count, dtype: int64

Column: cellularity
Type: object
Missing: 43
Unique Va

In [54]:

# Step 7: Group the features by their type

spec = {
    'clin_numeric': [
        'age_at_diagnosis',
        'tumor_size',
        'lymph_nodes_examined_positive',
        'neoplasm_histologic_grade'
    ],

    'clin_numeric_flag': [
        'tumor_stage'
    ],  

    'clin_categorical': [
        'type_of_breast_surgery',
        'cancer_type_detailed',
        'cellularity',
        'er_status',
        'her2_status',
        'pr_status',
        'inferred_menopausal_state',
        'primary_tumor_laterality'
    ],

    'mol_numeric': ['mutation_count'] + expr_block,

    'mol_categorical': [
        'pam50_+_claudin-low_subtype',
        'integrative_cluster',
        '3-gene_classifier_subtype',
        'her2_status_measured_by_snp6'
    ],

    'mut': mut_cols
}

all_spec_cols = [
    column
    for group in spec.values()
    for column in group
]

assert len(all_spec_cols) == len(set(all_spec_cols))
assert set(all_spec_cols) == set(COMBINED_FEATS)

print("Total features in groups:", len(all_spec_cols))

# Save the feature groups for later notebooks
with open('../configs/feature_spec.json', 'w') as file:
    json.dump(spec, file, indent=2)

# Make src a Python package
Path('../src').mkdir(parents=True, exist_ok=True)
Path('../src/__init__.py').touch()


Total features in groups: 680


In [55]:

%%writefile ../src/preprocessing.py

"""Preprocessing pipelines for the METABRIC survival project.

Every step (imputation, scaling, encoding, rare-mutation filter) is fitted
on training data only, because it lives inside a scikit-learn pipeline.
"""

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import VarianceThreshold


# Remove mutation genes that are too rare in the training data
MIN_MUT_RATE = 0.01
MUT_VARIANCE = MIN_MUT_RATE * (1 - MIN_MUT_RATE)

# Categories with fewer than 10 training examples can be grouped together
MIN_CATEGORY_COUNT = 10


def _numeric():
    """Fill missing numbers and scale numerical features."""
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])


def _numeric_with_flag():
    """Fill missing numbers and add an indicator for missing values."""
    return Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
    ])


def _categorical():
    """Fill missing categories and convert them into numerical columns."""
    return Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="infrequent_if_exist",
            min_frequency=MIN_CATEGORY_COUNT,
            sparse_output=False
        )),
    ])


def _mutation():
    """Remove mutation columns with very low variance."""
    return Pipeline([
        ("rare_filter", VarianceThreshold(threshold=MUT_VARIANCE)),
    ])


def build_preprocessor(feature_set, spec):
    """
    Build a preprocessing pipeline for clinical, molecular or combined data.
    The feature groups come from feature_spec.json.
    """
    parts = []

    if feature_set in ("clinical", "combined"):
        parts.append(("clin_num", _numeric(), spec["clin_numeric"]))
        parts.append((
            "clin_num_flag",
            _numeric_with_flag(),
            spec["clin_numeric_flag"]
        ))
        parts.append((
            "clin_cat",
            _categorical(),
            spec["clin_categorical"]
        ))

    if feature_set in ("molecular", "combined"):
        parts.append(("mol_num", _numeric(), spec["mol_numeric"]))
        parts.append((
            "mol_cat",
            _categorical(),
            spec["mol_categorical"]
        ))
        parts.append(("mut", _mutation(), spec["mut"]))

    if not parts:
        raise ValueError(
            "feature_set must be clinical, molecular or combined"
        )

    preprocessor = ColumnTransformer(
        parts,
        remainder="drop",
        verbose_feature_names_out=True
    )

    preprocessor.set_output(transform="pandas")

    return preprocessor


Overwriting ../src/preprocessing.py


In [56]:

# Step 8: Test the preprocessing pipelines

import sys
sys.path.append('..')

from src.preprocessing import build_preprocessor

with open('../configs/feature_spec.json') as file:
    spec = json.load(file)

for name in ['clinical', 'molecular', 'combined']:

    preprocessor = build_preprocessor(name, spec)

    # Fit and transform using training data only
    X_out = preprocessor.fit_transform(
        X_train[COMBINED_FEATS]
    )

    mutation_columns = [
        column for column in X_out.columns
        if column.startswith('mut__')
    ]

    merged_categories = [
        column for column in X_out.columns
        if 'infrequent' in column
    ]

    missing_flags = [
        column for column in X_out.columns
        if 'missingindicator' in column
    ]

    print("\nFeature set:", name)
    print("Output shape:", X_out.shape)
    print("Missing values after preprocessing:",
          int(X_out.isna().sum().sum()))
    print("Mutation genes kept:",
          len(mutation_columns), "of", len(mut_cols))
    print("Merged rare categories:", merged_categories)
    print("Missing-value flags:", missing_flags)

    assert X_out.isna().sum().sum() == 0
    assert X_out.shape[0] == len(X_train)



Feature set: clinical
Output shape: (1522, 27)
Missing values after preprocessing: 0
Mutation genes kept: 0 of 173
Merged rare categories: ['clin_cat__cancer_type_detailed_infrequent_sklearn']
Missing-value flags: ['clin_num_flag__missingindicator_tumor_stage']

Feature set: molecular
Output shape: (1522, 629)
Missing values after preprocessing: 0
Mutation genes kept: 113 of 173
Merged rare categories: ['mol_cat__pam50_+_claudin-low_subtype_infrequent_sklearn', 'mol_cat__her2_status_measured_by_snp6_infrequent_sklearn']
Missing-value flags: []

Feature set: combined
Output shape: (1522, 656)
Missing values after preprocessing: 0
Mutation genes kept: 113 of 173
Merged rare categories: ['clin_cat__cancer_type_detailed_infrequent_sklearn', 'mol_cat__pam50_+_claudin-low_subtype_infrequent_sklearn', 'mol_cat__her2_status_measured_by_snp6_infrequent_sklearn']
Missing-value flags: ['clin_num_flag__missingindicator_tumor_stage']


In [57]:

# Step 9: Create and save the cross-validation splits

from sklearn.model_selection import RepeatedStratifiedKFold

# Use 5 folds and repeat the split 3 times
cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=3,
    random_state=SEED
)

folds = []

for train_rows, val_rows in cv.split(
    X_train, y_train['event']
):
    folds.append({
        'train': train_rows.tolist(),
        'val': val_rows.tolist()
    })

# Save all folds for later notebooks
with open('../configs/cv_folds.json', 'w') as file:
    json.dump(folds, file, indent=2)

# Check the number of folds and event rates
validation_rates = [
    y_train['event'].iloc[fold['val']].mean()
    for fold in folds
]

print("Total folds:", len(folds))
print(
    "First fold - training rows:", len(folds[0]['train']),
    "| validation rows:", len(folds[0]['val'])
)
print(
    "Validation event rate range:",
    round(min(validation_rates), 3),
    "to",
    round(max(validation_rates), 3)
)


Total folds: 15
First fold - training rows: 1217 | validation rows: 305
Validation event rate range: 0.579 to 0.58


## Decision log (notebook 02)
- Outcome: time = overall_survival_months, event = (overall_survival == 0). This is overall survival, not cancer-specific.
- Patient 284 removed (censored at time 0): no information for the Cox likelihood, can break IPCW metrics. n = 1903.
- Mutation columns read as strings; '0' = no mutation, anything else = mutated (binary). 728 cells hold two variants.
- Excluded: patient_id, outcome columns (leakage); chemo/hormone/radio (not known at prediction time, confounding by indication); NPI (formula of included variables; reference baseline only); cohort (metadata for LOCO/subgroups); duplicates (cancer_type, oncotree_code, er_status_measured_by_ihc, tumor_other_histologic_subtype).
- Feature sets: clinical (13), molecular (667), combined (680).
- Split: 80/20, stratified on event, seed 42, patient level. Test set not touched until notebook 06.
- Pipelines: median/most-frequent imputation, missing flag for tumor_stage, one-hot with categories <10 merged, standard scaling, rare-mutation filter (~1%). All fitted on training data only.
- CV: 5-fold x 3 repeats on the training set, saved in configs/cv_folds.json.
- Limitation: er_status derivation not verified against dataset documentation (118 of 1874 patients disagree with the IHC version).